# Fine-tuning

### Configuração de ambiente

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '2'  # ou o ID da sua GPU
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = ''

### Imports

In [3]:
from os.path import join
from json import load, dump
from datetime import timedelta

from unsloth import FastVisionModel
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
from tqdm.notebook import tqdm

import torch

from scripts.authentication import authenticate_huggingface
from scripts.data import SimpleLesionData, SimpleDatasetAnalysis
from scripts.messages import create_training_message
from scripts.training import Training

import scripts.definitions as defs

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


### Autenticação

In [4]:
authenticate_huggingface()

Insira seu token do Hugging Face:  ········


### Configuração

In [5]:
VERSION = 'V7'

training_hyperparameters = Training(
    base_model_name=defs.BASE_MODEL_NAME,
    trained_model_name=defs.MODEL_NAME,
    quantization=True,
    prompt_type=defs.PromptType.REPORT,
    version=VERSION,
    size=11,
    peft_hyperparameters={
        # Camadas
        'finetune_vision_layers': True,
        'finetune_language_layers': True,
        'finetune_attention_modules': True,
        'finetune_mlp_modules': True,
        # LoRA
        'r': 128,
        'lora_alpha': 128,
        'lora_dropout': 0.1,
        'bias': 'none',
        'random_state': defs.STATIC_RANDOM_STATE,
        'use_rslora': True,
        'loftq_config': None
    },
    sft_hyperparameters={
        # Controle de memória
        'per_device_train_batch_size': 4,
        'gradient_accumulation_steps': 1,
        # Controle de treinamento
        'learning_rate': 1e-4,
        'weight_decay': 0.01,
        'num_train_epochs': 2.0,
        'lr_scheduler_type': 'cosine',
        'warmup_ratio': 0.1,
        'optim': 'paged_adamw_32bit',
        # Monitoramento
        'logging_steps': 1,
        'report_to': 'tensorboard',
        'output_dir': 'outputs',
        # Aleatoriedade
        'seed': defs.STATIC_RANDOM_STATE,
        # Tipos
        'bf16': is_bf16_supported(),
        'fp16': not is_bf16_supported(),
        # Dataset
        'remove_unused_columns': False,
        'dataset_text_field': '',
        'dataset_kwargs': {'skip_prepare_dataset': True},
        'dataset_num_proc': 4,
        # Janela de contexto
        'max_seq_length': defs.MAX_TOKENS
    },
    used_memory=0.0,
    training_time=0.0
)

with open(join(defs.TRAINING_PATH, f'hyperparameters_{training_hyperparameters.version}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

### Carregamento do dataset

In [6]:
with open(join(defs.DATA_PATH, 'stt_data', 'training_dataset.json'), 'r', encoding='utf-8') as file:
    training_dataset = [SimpleLesionData(**data) for data in load(file)]

with open(join(defs.DATA_PATH, 'training_dataset_analysis.json'), 'r', encoding='utf-8') as file:
    training_dataset_analysis = SimpleDatasetAnalysis(**load(file))

### Preparação das mensagens

In [7]:
training_messages = []
validation_messages = []

for lesion_data in tqdm(training_dataset, desc='Criando mensagens de treinamento: '):
    training_messages.append(create_training_message(training_hyperparameters.prompt_type,
                                                     lesion_data,
                                                     training_dataset_analysis))

Criando mensagens de treinamento:   0%|          | 0/15664 [00:00<?, ?it/s]

### Inicialização do LLaMa 3.2

In [8]:
model, tokenizer = FastVisionModel.from_pretrained(
    training_hyperparameters.base_model_name,
    load_in_4bit=training_hyperparameters.quantization,
    use_gradient_checkpointing='unsloth'
)

==((====))==  Unsloth 2026.5.8: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 1. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

### Configuração de treinamento

In [9]:
model = FastVisionModel.get_peft_model(
    model,
    **training_hyperparameters.peft_hyperparameters
)

FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=training_messages,
    args=SFTConfig(**training_hyperparameters.sft_hyperparameters),
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: You set `max_seq_length` as 2190 but the maximum the model supports is 2048. We shall reduce it.


### Treinamento

In [10]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'bos_token_id': 128000}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,664 | Num Epochs = 2 | Total steps = 7,832
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 1 x 1) = 4
 "-____-"     Trainable parameters = 537,395,200 of 11,207,616,035 (4.79% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,5.631549
2,5.357753
3,5.486254
4,5.326551
5,4.910587
6,5.027645
7,4.802268
8,4.284544
9,3.630682
10,3.138769


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-2500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-3500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-4000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-4500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-5000/tokenizer_config.json.
Unsloth: Restored add

In [11]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
print(f'Tempo de treinamento: {timedelta(seconds=trainer_stats.metrics["train_runtime"])}')
print(f'Memória máxima reservada: {used_memory} GB')

training_hyperparameters.used_memory = used_memory
training_hyperparameters.training_time = trainer_stats.metrics['train_runtime']

hyperparameters_name = f'hyperparameters_{training_hyperparameters.version}'

if training_hyperparameters.quantization:
    hyperparameters_name += '-4bit'

with open(join(defs.TRAINING_PATH, f'{hyperparameters_name}.json'), 'w', encoding='utf-8') as file:
    dump(training_hyperparameters.model_dump(), file, indent=4, ensure_ascii=False)

Tempo de treinamento: 6:09:43.654900
Memória máxima reservada: 17.98 GB


### Salvamento

In [16]:
import shutil
from os import makedirs
from os.path import join, exists
from json import load, dump

# Primeiro lista os arquivos disponíveis no checkpoint
import os
checkpoint_path = 'outputs/checkpoint-18525'
print(os.listdir(checkpoint_path))

['README.md', 'processor_config.json', 'training_args.bin', 'tokenizer_config.json', 'adapter_config.json', 'adapter_model.safetensors', 'tokenizer.json', 'optimizer.pt', 'trainer_state.json', 'rng_state.pth', 'chat_template.jinja', 'scheduler.pt']


In [ ]:
RECUPERACAO V1

In [18]:


import shutil
from os import makedirs
from os.path import join
from json import load, dump

trained_model_name = 'LLaDerm-V1-11B-4bit'
save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
checkpoint_path = 'outputs/checkpoint-18525'

makedirs(save_path, exist_ok=True)

files_to_copy = [
    'adapter_config.json',
    'adapter_model.safetensors',
    'tokenizer_config.json',
    'tokenizer.json',
    'processor_config.json',
    'chat_template.jinja',
]

for filename in files_to_copy:
    shutil.copy2(join(checkpoint_path, filename), join(save_path, filename))
    print(f'Copiado: {filename}')

print(f'\nModelo salvo em: {save_path}')

from json import loads, dump

models_path = join(defs.TRAINING_PATH, 'models.json')
models = {}

with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
    if content:
        models = {name: defs.Model(**data) for name, data in loads(content).items()}

new_model = defs.Model(
    local=True,
    quantized=training_hyperparameters.quantization,
    prompt_type=training_hyperparameters.prompt_type,
    version=training_hyperparameters.version,
    size=training_hyperparameters.size
)

models[trained_model_name] = new_model

for name, m in models.items():
    models[name] = m.model_dump() if hasattr(m, 'model_dump') else m

with open(join(defs.TRAINING_PATH, 'models.json'), 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V1-11B-4bit
Registrado em models.json


In [15]:
model, tokenizer = FastVisionModel.from_pretrained(
    'outputs/checkpoint-18525',
    load_in_4bit=training_hyperparameters.quantization,
    use_gradient_checkpointing=False,
)

==((====))==  Unsloth 2026.5.5: Fast Mllama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 80GB HBM3. Num GPUs = 8. Max memory: 79.189 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 79.19 GiB of which 21.81 MiB is free. Process 44721 has 1.13 GiB memory in use. Process 572518 has 6.77 GiB memory in use. Process 3753065 has 63.97 GiB memory in use. Process 1248290 has 7.27 GiB memory in use. Of the allocated memory 6.60 GiB is allocated by PyTorch, and 4.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
SALVAMENTO SEM KERNEL CAIR

In [12]:
trained_model_name = f'{training_hyperparameters.trained_model_name}-{training_hyperparameters.version}-{training_hyperparameters.size}B'

if training_hyperparameters.quantization:
    trained_model_name += '-4bit'

if training_hyperparameters.prompt_type == defs.PromptType.SIMPLE_CLASSIFICATION:
    trained_model_name += '-SC'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

models_path = join(defs.TRAINING_PATH, 'models.json')
models = {}

with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
    if content:
        models = {name: defs.Model(**data) for name, data in loads(content).items()}


new_model = defs.Model(
    local=True,
    quantized=training_hyperparameters.quantization,
    prompt_type=training_hyperparameters.prompt_type,
    version=training_hyperparameters.version,
    size=training_hyperparameters.size
)

models[trained_model_name] = new_model

for name, trained_model in models.items():
    models[name] = trained_model.model_dump()  # type: ignore

with open(join(defs.TRAINING_PATH, 'models.json'), 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

Unsloth: Restored added_tokens_decoder metadata in ../results/adapter_weights/LLaDerm-V4-11B-4bit/tokenizer_config.json.


NameError: name 'loads' is not defined

In [ ]:
RECUPERACAO V2

In [2]:
import shutil
from os import makedirs
from os.path import join
from json import loads, dump
import scripts.definitions as defs

trained_model_name = 'LLaDerm-V2-11B-4bit'
save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
checkpoint_path = 'outputs/checkpoint-7182'

makedirs(save_path, exist_ok=True)

for filename in ['adapter_config.json', 'adapter_model.safetensors',
                 'tokenizer_config.json', 'tokenizer.json',
                 'processor_config.json', 'chat_template.jinja']:
    shutil.copy2(join(checkpoint_path, filename), join(save_path, filename))
    print(f'Copiado: {filename}')

print(f'\nModelo salvo em: {save_path}')

# Registra no models.json
models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()

models = loads(content) if content else {}
models[trained_model_name] = {
    'local': True,
    'quantized': True,
    'prompt_type': 'report',
    'version': 'V2',
    'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V2-11B-4bit
Registrado em models.json


In [ ]:
RECUPERACAO V3

In [2]:
import shutil
from os import makedirs
from os.path import join
from json import loads, dump
import scripts.definitions as defs

trained_model_name = 'LLaDerm-V3-11B-4bit'
save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
checkpoint_path = 'outputs/checkpoint-17955'

makedirs(save_path, exist_ok=True)

for filename in ['adapter_config.json', 'adapter_model.safetensors',
                 'tokenizer_config.json', 'tokenizer.json',
                 'processor_config.json', 'chat_template.jinja']:
    shutil.copy2(join(checkpoint_path, filename), join(save_path, filename))
    print(f'Copiado: {filename}')

print(f'\nModelo salvo em: {save_path}')

models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()

models = loads(content) if content else {}
models[trained_model_name] = {
    'local': True,
    'quantized': True,
    'prompt_type': 'report',
    'version': 'V3',
    'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V3-11B-4bit
Registrado em models.json


In [13]:
from json import loads, dump
import scripts.definitions as defs
from os.path import join

trained_model_name = 'LLaDerm-V4-11B-4bit'
models_path = join(defs.TRAINING_PATH, 'models.json')

with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()

models = loads(content) if content else {}
models[trained_model_name] = {
    'local': True,
    'quantized': True,
    'prompt_type': 'report',
    'version': 'V4',
    'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print('Registrado em models.json')


Registrado em models.json


In [ ]:
V5

In [1]:
import shutil
from os import makedirs
from os.path import join
from json import loads, dump
import scripts.definitions as defs

trained_model_name = 'LLaDerm-V5-11B-4bit'
checkpoint_path = 'outputs/checkpoint-12734'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
makedirs(save_path, exist_ok=True)

for filename in ['adapter_config.json', 'adapter_model.safetensors',
                 'tokenizer_config.json', 'tokenizer.json',
                 'processor_config.json', 'chat_template.jinja']:
    src = join(checkpoint_path, filename)
    try:
        shutil.copy2(src, join(save_path, filename))
        print(f'Copiado: {filename}')
    except FileNotFoundError:
        print(f'Não encontrado (pode ignorar): {filename}')

models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
models = loads(content) if content else {}

models[trained_model_name] = {
    'local': True, 'quantized': True,
    'prompt_type': 'report', 'version': 'V5', 'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print(f'\nModelo salvo em: {save_path}')
print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V5-11B-4bit
Registrado em models.json


In [ ]:
V6

In [1]:
import shutil
from os import makedirs
from os.path import join
from json import loads, dump
import scripts.definitions as defs

trained_model_name = 'LLaDerm-V6-11B-4bit'
checkpoint_path = 'outputs/checkpoint-10478'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
makedirs(save_path, exist_ok=True)

for filename in ['adapter_config.json', 'adapter_model.safetensors',
                 'tokenizer_config.json', 'tokenizer.json',
                 'processor_config.json', 'chat_template.jinja']:
    src = join(checkpoint_path, filename)
    try:
        shutil.copy2(src, join(save_path, filename))
        print(f'Copiado: {filename}')
    except FileNotFoundError:
        print(f'Não encontrado (pode ignorar): {filename}')

models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
models = loads(content) if content else {}

models[trained_model_name] = {
    'local': True, 'quantized': True,
    'prompt_type': 'report', 'version': 'V6', 'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print(f'\nModelo salvo em: {save_path}')
print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V6-11B-4bit
Registrado em models.json


In [ ]:
V7

In [12]:
import shutil
from os import makedirs
from os.path import join
from json import loads, dump
import scripts.definitions as defs

trained_model_name = 'LLaDerm-V7-11B-4bit'
checkpoint_path = 'outputs/checkpoint-7832'

save_path = join(defs.RESULTS_PATH, 'adapter_weights', trained_model_name)
makedirs(save_path, exist_ok=True)

for filename in ['adapter_config.json', 'adapter_model.safetensors',
                 'tokenizer_config.json', 'tokenizer.json',
                 'processor_config.json', 'chat_template.jinja']:
    src = join(checkpoint_path, filename)
    try:
        shutil.copy2(src, join(save_path, filename))
        print(f'Copiado: {filename}')
    except FileNotFoundError:
        print(f'Não encontrado (pode ignorar): {filename}')

models_path = join(defs.TRAINING_PATH, 'models.json')
with open(models_path, 'r', encoding='utf-8') as file:
    content = file.read().strip()
models = loads(content) if content else {}

models[trained_model_name] = {
    'local': True, 'quantized': True,
    'prompt_type': 'report', 'version': 'V7', 'size': 11
}

with open(models_path, 'w', encoding='utf-8') as file:
    dump(models, file, indent=4, ensure_ascii=False)

print(f'\nModelo salvo em: {save_path}')
print('Registrado em models.json')


Copiado: adapter_config.json
Copiado: adapter_model.safetensors
Copiado: tokenizer_config.json
Copiado: tokenizer.json
Copiado: processor_config.json
Copiado: chat_template.jinja

Modelo salvo em: ../results/adapter_weights/LLaDerm-V7-11B-4bit
Registrado em models.json
